# 00 Colab Bootstrap

Generated by `scripts/revision/create_revision_notebooks.py`.


## Purpose

Use this first when opening Colab directly. It mounts Drive, checks large artifacts, then clones or updates the GitHub branch used as the code transport layer.

In [ ]:
from pathlib import Path
import os
import subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_DIR = DRIVE_ROOT / 'ECG-RAMBA'
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = 'main'
os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)

print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)


## Verify Drive Artifacts

In [ ]:
required_files = [
    'WFDB-ChapmanShaoxing.zip',
    'PTB-XL.zip',
    'Georgia.zip',
]

missing = []
for name in required_files:
    path = DRIVE_ROOT / name
    print(f'{name}: exists={path.exists()} path={path}')
    if not path.exists():
        missing.append(str(path))

model_dir = DRIVE_ROOT / 'model'
fold_ckpts = sorted(model_dir.glob('fold*_best.pt'))
print('\nmodel_dir       :', model_dir)
print('fold checkpoints:', len(fold_ckpts))
print('fold PCA manifest:', Path('reports/revision/manifests/fold_pca_manifest.json').exists())

if len(fold_ckpts) < 5:
    missing.append(f'Expected 5 fold*_best.pt files under {model_dir}')
if missing:
    raise FileNotFoundError('Missing required Drive artifacts:\n' + '\n'.join(missing))


## Clone Or Update Repo

In [ ]:
def run(cmd, cwd=None):
    print(f'$ {cmd}')
    subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None, check=True)

if (REPO_DIR / '.git').exists():
    run('git fetch origin', cwd=REPO_DIR)
    run(f'git checkout {BRANCH}', cwd=REPO_DIR)
    run(f'git pull --ff-only origin {BRANCH}', cwd=REPO_DIR)
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f'{REPO_DIR} exists but is not a git repo. Rename it or set REPO_DIR to another folder.')
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
run('git status --short --branch')


## Install Lightweight Dependencies

In [ ]:
!python --version
!pip install -q "numpy>=2.0,<2.1" "scipy>=1.14.1,<2.0" "pandas==2.2.2" "scikit-learn>=1.2,<1.9" "requests==2.32.4" "pillow>=8,<12" tqdm "wfdb==4.1.2" joblib matplotlib seaborn packaging neurokit2 iterative-stratification thop


## Run A0 Audit

In [ ]:
os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.chdir(REPO_DIR)
!python scripts/revision/00_audit_protocol.py
